# CR-2 G1 の前の受け入れ 1：試験と人工データ（API なし・CPU）

凍結した計画の「G1 の前の受け入れ」のうち、**データも API も使わない部分**です。

- **試験**：CR-2 の試験（a）〜（p）と、CR-1・CR-1a・下見 3 の回帰試験を全部回します。
- **人工データの安全診断**：差がない世界 9 個を、各 2,000 データ・並べ替え 199 回で回します。全試行の `BEYOND_*` の片側 95% の上側の限界が **0.07 以下**なら、その世界は合格。9 世界すべて合格で合格です。
- **既知の合格例の回帰確認**：CHASE-Math に寄せた差なしの 3 世界（計画を書く前に合格を見たもの）を同じ乱数種で回し直し、合格が再現されることを確かめます（独立の確かめではありません。数えるのは 9 世界だけ）。
- 記録と数が同じかどうかも書きます（numpy の版で細部が変わりうるので、止める条件にはしません）。

**使い方**：CPU のランタイム。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で **30〜50 分**。結果は Drive の `DECISION_POINTS/cr2/PREG1/CR2_preg1_result.json`。終わったら Claude に教えてね（Claude がコードの固定 `notes/cr2_code_lock.md` を書きます）。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, ancestors, (pinned SDK and key, never printed), notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
EXPECTED_HEAD = "a9fd5344084cdec8c402f1768cdf993056a7b86f"
ANCESTORS = []
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
for anc in ["7949d74"] + ANCESTORS:
    assert subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", anc, head]).returncode == 0, anc + " が祖先ではありません"
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from pork import cr2 as C
from pork import cr2_search as S
from pork import cr2_run as RN
from pork import cr2_g2 as G
from pork import cr2_g3 as G3
from pork import cr2_stage3b as T3
from pork import cr1_run as R1
from pork import alt_prelook as P
import numpy
RN.check_prompt()
D = Path("/content/drive/MyDrive/DECISION_POINTS/cr2")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))

def git_blob(path):
    return subprocess.check_output(["git", "hash-object", path], text=True).strip()

def write_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    R1._atomic_write(path, obj)
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_chase_verified():
    """Plan 1. (H4, I3): pinned revision, canonical hash from the Hugging Face API vs the downloaded bytes."""
    from huggingface_hub import HfApi, hf_hub_download
    info = HfApi().dataset_info(C.DATASET_ID, revision=C.DATASET_REVISION, files_metadata=True, token=False)
    sib = next((s for s in info.siblings if s.rfilename == C.DATASET_FILE), None)
    meta = None
    if sib is not None:
        lfs = sib.lfs
        if lfs is not None and not isinstance(lfs, dict):
            lfs = {"sha256": getattr(lfs, "sha256", None), "size": getattr(lfs, "size", None)}
        meta = {"blob_id": sib.blob_id, "lfs": lfs}
    p = hf_hub_download(C.DATASET_ID, C.DATASET_FILE, repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    data = Path(p).read_bytes()
    prov = C.check_provenance(data, meta, info.sha)
    rp = hf_hub_download(C.DATASET_ID, "README.md", repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    readme = Path(rp).read_text(encoding="utf-8")
    rows = C.parse_rows(data)
    del data
    return rows, prov, {"readme_sha256": C.sha256_text(readme), "license": C.readme_license(readme)}

def rebuild_stage2(file_sha=None, cand_sha=None, excl_sha=None):
    rows, prov, readme = load_chase_verified()
    if file_sha is not None:
        assert prov["file_sha256"] == file_sha, "データのファイルが段2 の固定と違います。止めます（1 件も送りません）。"
    pre = C.reproduce_prelook3(rows, P.load_math())
    cand = C.build_candidates(rows, pre)
    if cand_sha is not None:
        assert cand["candidates_sha256"] == cand_sha, "候補の一覧が段2 の固定と違います。止めます。"
    if excl_sha is not None:
        assert cand["excluded_sha256"] == excl_sha, "除外した行が段2 の固定と違います。止めます。"
    by_id = {int(r["ID"]): r for r in rows}
    cands = [{**c, "question": C.problem_text(by_id[c["id"]]), "gold": str(C.gold(by_id[c["id"]]))}
             for c in cand["candidates"]]                                     # PRIVATE, memory only
    del rows, by_id
    return cand, cands, prov, readme, pre

print("study:", C.STUDY, "| exact head:", head, "| numpy", numpy.__version__, "| python", sys.version.split()[0])

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-2 G1 前の受け入れ 1：試験と人工データ（API なし・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-2 pre-G1 tests and sims', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — tests (a)-(p) and regressions.  Nothing is sent anywhere.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "scipy"], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
TESTS = ["tests/test_cr2.py", "tests/test_cr2_runs.py", "tests/test_cr2_search.py", "tests/test_alt_prelook.py",
         "tests/test_cr1.py", "tests/test_cr1_g2.py", "tests/test_cr1_g3.py", "tests/test_cr1_items.py",
         "tests/test_cr1_requests.py", "tests/test_cr1_run.py"]
missing_tests = [t for t in TESTS if not Path(t).exists()]
assert not missing_tests, "試験のファイルがありません。止めます: " + ", ".join(missing_tests)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", *TESTS], capture_output=True, text=True,
                   env=dict(os.environ, PYTHONPATH=str(REPO)))
TEST_LINE = r.stdout.strip().splitlines()[-1]
print(TEST_LINE)
assert r.returncode == 0, "試験が通りません。止めます。\n" + r.stdout[-3000:]

In [ ]:
# Cell 3 — synthetic safety diagnostic (9 null worlds) and the regression check (3 known CHASE-shaped null worlds).
t0 = time.time()
def run_shards(cmd, n=2):
    ps = [subprocess.Popen([sys.executable] + cmd + [f"{i}/{n}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
          for i in range(n)]
    lines = []
    for p in ps:
        out, err = p.communicate()
        assert p.returncode == 0, err[-2000:]
        lines += [l for l in out.splitlines() if l.startswith("idx=")]
    return sorted(lines, key=lambda l: int(l.split("|")[0].split("=")[1]))

def parse(line):
    d = {}
    for part in line.split(" | "):
        k, _, val = part.partition("=")
        d[k.strip()] = val.strip()
    return d

SAFETY = run_shards(["scripts/cr2_s2_recalibration.py", "2000", "0", "199"])
REGRESS = run_shards(["scripts/cr2_chase_power.py", "2000", "0", "199"])
assert len(SAFETY) == 9 and len(REGRESS) == 3, (len(SAFETY), len(REGRESS))
rec_s = [l for l in (REPO / "reports/cr2_s2_recalibration.txt").read_text().splitlines() if l.startswith("idx=")][:9]
rec_r = [l for l in (REPO / "reports/cr2_chase_power.txt").read_text().splitlines() if l.startswith("idx=")][:3]
worlds = []
for name, got, rec in (("safety", SAFETY, rec_s), ("regression", REGRESS, rec_r)):
    for g_, r_ in zip(got, rec):
        d = parse(g_)
        worlds.append({"kind": name, "world": d["world"], "beyond_count": int(d["beyond_count"]), "n": int(d["n"]),
                       "cp_upper_95_one_sided": float(d["cp_upper_95_one_sided"]), "accept": d["accept"] == "True",
                       "same_as_record": g_ == r_})
for w in worlds:
    print(w["kind"], w["world"], "| BEYOND", w["beyond_count"], "/", w["n"], "| 上側の限界 %.4f" % w["cp_upper_95_one_sided"],
          "| 合格" if w["accept"] else "| 不合格", "| 記録と同じ:", w["same_as_record"])
safety_pass = all(w["accept"] for w in worlds if w["kind"] == "safety")
regress_pass = all(w["accept"] for w in worlds if w["kind"] == "regression")
RESULT = {"schema": "cr2-preg1-v1", "study": C.STUDY, "code_head": head, "numpy": numpy.__version__,
          "python": sys.version.split()[0], "tests": TEST_LINE, "tests_passed": True, "worlds": worlds,
          "safety_9_worlds_pass": safety_pass, "regression_3_worlds_pass": regress_pass,
          "pass": bool(safety_pass and regress_pass), "seconds": round(time.time() - t0, 1),
          "note": "safety = 9 null worlds (the gate); regression = 3 CHASE-shaped null worlds already seen before v9 (not independent)"}
sha = write_json(D / "PREG1/CR2_preg1_result.json", RESULT)
print("安全診断（9 世界）:", "合格" if safety_pass else "不合格", "| 回帰確認（3 世界）:", "再現" if regress_pass else "再現せず",
      "| 結果ファイルの sha256:", sha)
assert RESULT["pass"], "人工データの確かめに落ちました。G1 には進みません。"
notify('終わりました：CR-2 の G1 前の受け入れ 1（試験と人工データ）が合格しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()